In [5]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import StratifiedKFold

In [6]:
# 1. Load files
train_all = pd.read_csv("./train.csv")
test_ids = pd.read_csv("./test.csv")
sample = pd.read_csv("./sample.csv")

print("train.csv shape:", train_all.shape)
print("test.csv shape:", test_ids.shape)
print("sample.csv shape:", sample.shape)

display(train_all.head())
display(test_ids.head())

train.csv shape: (139753, 9)
test.csv shape: (13976, 2)
sample.csv shape: (13976, 2)


,Id,ProductId,UserId,HelpfulnessNumerator,HelpfulnessDenominator,Time,Summary,Text,Score
0,1049849,B000MR9D5E,A1EKSETIBS9ETQ,0,0,1198281600,"Great nature series, but not all scenes looked...",I have watched numbers of nature shows and ser...,4.0
1,999834,B000GAKFIG,AR0HFYHYHDGQQ,2,5,1245024000,Agatha Christie's Marple: Series 2,As a devoted fan of all of Agatha Christie's f...,5.0
2,218826,6300215776,A37S3ACL57LN62,11,15,1126137600,Childish Entertainment,This movie is about a script writer and a secr...,2.0
3,796384,B00019071C,A1TO1P3NV7OAU6,2,2,1351036800,The weakest Babylon 5 season,This is the weakest Babylon 5 season. After wi...,4.0
4,1219784,B001NFNFN0,ATCM1W7HWIC6U,0,0,1381708800,Versatile and effective,This video will always have a sweet spot on my...,5.0


,Id,Score
0,1224650,NaN
1,1019381,NaN
2,504719,NaN
3,1622425,NaN
4,482286,NaN


In [7]:
# 2. Split labeled and unlabeled rows from train.csv
train_df = train_all[train_all["Score"].notna()].copy()
hidden_test_df = train_all[train_all["Score"].isna()].copy()

train_df["Score"] = train_df["Score"].astype(int)

print("usable training rows:", train_df.shape)
print("rows to predict from train.csv:", hidden_test_df.shape)

print("\nTraining label distribution:")
print(train_df["Score"].value_counts().sort_index())

usable training rows: (125777, 9)
rows to predict from train.csv: (13976, 9)

Training label distribution:
Score
1     7593
2     7567
3    14857
4    28572
5    67188
Name: count, dtype: int64


In [8]:
# 3. Build the actual prediction table

pred_df = test_ids[["Id"]].merge(
    hidden_test_df,
    on="Id",
    how="left",
    validate="one_to_one"
)

print("prediction dataframe shape:", pred_df.shape)
print("missing rows after merge:", pred_df["Id"].isna().sum())

display(pred_df.head())

prediction dataframe shape: (13976, 9)
missing rows after merge: 0


,Id,ProductId,UserId,HelpfulnessNumerator,HelpfulnessDenominator,Time,Summary,Text,Score
0,1224650,B001OQCV6A,A3HUN3KP383B23,0,1,1395619200,Sherlock Holmes,I bought this thinking it was Sherlock Holmes ...,NaN
1,1019381,B000I8OFLO,A2VJ80PM1G00QV,2,2,1222300800,"FINE SINGING, DESTRUCTIVE STAGING",1. Scenic design and the Directors use of it p...,NaN
2,504719,B00000IQCC,A2EGK0YRDF4ZZB,11,11,1018310400,Moving story of Jesus's message in a modern re...,J&eacute;sus de Montr&eacute;al was a stunni...,NaN
3,1622425,B00B5UBDA0,A21I62TCDL4754,0,0,1372118400,Paul Anka has aged well and performs well,I was looking for Blu Ray concerts for my new ...,NaN
4,482286,6305892946,A1IQ9E6I3PIUFF,1,2,969321600,Great end to the series.Or is it???????????,Despite what everyone else says this movie is ...,NaN


In [9]:
# 4. Build text features using Summary + Text
for col in ["Summary", "Text"]:
    if col not in train_df.columns:
        train_df[col] = ""
    if col not in pred_df.columns:
        pred_df[col] = ""

train_df["combined"] = train_df["Summary"].fillna("") + " " + train_df["Text"].fillna("")
pred_df["combined"] = pred_df["Summary"].fillna("") + " " + pred_df["Text"].fillna("")

tfidf = TfidfVectorizer(
    max_features=100000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

X_train = tfidf.fit_transform(train_df["combined"])
y = train_df["Score"].astype(int).values
X_pred = tfidf.transform(pred_df["combined"])

print("X_train shape:", X_train.shape)
print("X_pred shape:", X_pred.shape)

X_train shape: (125777, 100000)
X_pred shape: (13976, 100000)


In [10]:
# 5. Ensemble: LogisticRegression + SGDClassifier
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

test_pred_log = np.zeros((X_pred.shape[0], 5))
test_pred_sgd = np.zeros((X_pred.shape[0], 5))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y), 1):
    X_tr, X_va = X_train[tr_idx], X_train[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    log_model = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        C=3.0,
        random_state=42
    )
    log_model.fit(X_tr, y_tr)
    test_pred_log += log_model.predict_proba(X_pred) / skf.n_splits

    sgd_model = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=1e-5,
        class_weight="balanced",
        max_iter=3000,
        tol=1e-4,
        random_state=42
    )
    sgd_model.fit(X_tr, y_tr)
    test_pred_sgd += sgd_model.predict_proba(X_pred) / skf.n_splits

    print(f"fold {fold} done")

fold 1 done
fold 2 done
fold 3 done
fold 4 done
fold 5 done


In [11]:
# 6. Final predictions
blend_proba = 0.7 * test_pred_log + 0.3 * test_pred_sgd
final_pred = np.argmax(blend_proba, axis=1) + 1

print("Prediction distribution:")
print(pd.Series(final_pred).value_counts().sort_index())

Prediction distribution:
1     964
2     859
3    1761
4    3328
5    7064
Name: count, dtype: int64


In [12]:
# 7. Build submission
submission = sample.copy()
submission["Id"] = test_ids["Id"].values
submission["Score"] = final_pred.astype(int)

print(submission["Score"].value_counts().sort_index())
display(submission.head())

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")

Score
1     964
2     859
3    1761
4    3328
5    7064
Name: count, dtype: int64


,Id,Score
0,1224650,5
1,1019381,4
2,504719,4
3,1622425,5
4,482286,5


Saved submission.csv
